In [1]:
import duckdb
import gspread
import pandas as pd
# import numpy as np
from google.oauth2.service_account import Credentials

SCOPES = [
    # 'https://www.googleapis.com/auth/bigquery',
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets'
    # ,'https://www.googleapis.com/auth/devstorage.full_control'
]

credentials = Credentials.from_service_account_file(r'.secrets\ads-txt-validator-BQ.json', scopes=SCOPES)

In [ ]:
# con = duckdb.connect("my_db.duckdb")

# # Set a temp folder with lots of space
# # con.execute("PRAGMA temp_directory='D:/duckdb_temp';")

# con.execute("SET memory_limit='2GB';")

# # Remove spill limit OR increase it
# con.execute("PRAGMA max_temp_directory_size='20GB';")  # or '20GB' or '0

# con.execute("SELECT * FROM duckdb_settings() WHERE name IN ('memory_limit', 'max_temp_directory_size')").fetchdf()


,name,value,description,input_type,scope
0,memory_limit,1.8 GiB,The maximum memory of the system (e.g. 1GB),VARCHAR,GLOBAL
1,max_temp_directory_size,18.6 GiB,The maximum amount of data stored inside the '...,VARCHAR,GLOBAL


In [2]:
sheet_client:gspread.Client = gspread.Client(auth=credentials)

def adhoc_gsheet_pull(worksheet_name:str):

        google_sheet_id = '1Vuul2-7WJX_E3qPahlZzurYr5hPIWzxrLW3ry2q78XI'
        # worksheet_name = 'main_file'

        spreadsheet = sheet_client.open_by_key(google_sheet_id)

        worksheet = spreadsheet.worksheet(worksheet_name)

        list_of_dicts = worksheet.get_all_records()

        df = pd.DataFrame(list_of_dicts)

        try:
            if "ORG_ID" in df.columns:
                df = df.astype({"ORG_ID":"int64"}).astype({"ORG_ID":"string"})
        except Exception as e:
            print(f"{e=}")

        # print(df.dtypes)

        return df

main_file_df = adhoc_gsheet_pull(worksheet_name = 'main_file')
# expected_ads_txt_df = adhoc_gsheet_pull(worksheet_name = 'ads_txt_lines')
# main_file_df

In [3]:
len(main_file_df)

3889

In [4]:
raw_tbl = duckdb.read_parquet(r"C:\Users\zaid\Desktop\Python\ads_txt_v2\data_output\2025-05-27\ads_txt_success_2025-05-27-12-42-59.parquet")
err_tbl = duckdb.read_parquet(r"C:\Users\zaid\Desktop\Python\ads_txt_v2\data_output\2025-05-27\ads_txt_failure_2025-05-27-12-42-59.parquet")

In [5]:
output_tbl = duckdb.sql(
    """
CREATE OR REPLACE MACRO SAFE_OFFSET(arr, idx) AS (
    CASE
        WHEN idx > 0 AND idx <= array_length(arr) THEN arr[idx]
        ELSE NULL
    END
);
WITH main_client_tbl AS (
  SELECT 
  Publisher AS publisher
  , ORG_ID AS org_id
  , CASE WHEN Inventory_Type = 'App - (app-ads.txt)' THEN 'app-ads.txt'
         WHEN Inventory_Type = 'Web - (ads.txt)' THEN 'ads.txt'
         ELSE Inventory_Type END AS inventory_type
  , Relationship_Type AS relationship_type
  , Account_Manager as account_manager_name
  , Account_Manager_Email as account_manager_email
  , Domain AS domain
  , Ad_Request AS ad_request
  , Revenue AS revenue
  , Expected_ads_txt_line as ads_txt_lines 
  FROM main_file_df
  WHERE 
  domain IS NOT NULL
  AND inventory_type IS NOT NULL
  AND org_id IS NOT NULL
  AND relationship_type IS NOT NULL
),

required_domains AS (
  SELECT DISTINCT domain, inventory_type, CONCAT(domain,'--',inventory_type) AS relevent_domains_key FROM main_client_tbl
),

required_bronze_data AS (
  SELECT LOWER(REGEXP_REPLACE(REPLACE(ads_txt_line, ' ', ''), '#.*','')) AS ads_txt_lines
  , domain
  , inventory_type
  , ads_txt_url AS ads_txt_url
  FROM raw_tbl
  WHERE 
  CONCAT(domain,'--',inventory_type) IN (SELECT DISTINCT relevent_domains_key from required_domains) 
  AND ads_txt_line is not null and  not starts_with(ads_txt_line, '#')
),

duplicated_rows_bronze_data AS (
  SELECT 
  ads_txt_lines
  , domain
  , inventory_type
  , ads_txt_url
  -- , execution_date 
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),1))  as ssp_domain
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),2))  as pub_id
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),3))  as relationship_type
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),4))  as ssp_id
  ,ROW_NUMBER() OVER (
      PARTITION BY 
        domain, 
        inventory_type, 
        ads_txt_url, 
        -- execution_date, 
        TRIM(SAFE_OFFSET(SPLIT(ads_txt_lines, ','),1)),  -- ssp_domain
        TRIM(SAFE_OFFSET(SPLIT(ads_txt_lines, ','),2)),  -- pub_id
        TRIM(SAFE_OFFSET(SPLIT(ads_txt_lines, ','),3))   -- relationship_type
      -- ORDER BY execution_date DESC  -- Keeping the latest record
    ) AS row_num
  FROM required_bronze_data
), 

cleaned_bronze_data AS (
  SELECT * FROM duplicated_rows_bronze_data WHERE row_num = 1
),


cleaned_pubs_data AS (
  SELECT publisher
  , org_id
  , inventory_type
  , relationship_type AS relationship_type_given
  , account_manager_name
  , account_manager_email
  , domain
  , ad_request
  , revenue
  , ads_txt_lines 
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),1))  as ssp_domain
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),2))  as pub_id
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),3))  as relationship_type
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),4))  as ssp_id
FROM main_client_tbl),

error_tbl AS (SELECT
    domain,
    inventory_type,
    repr_error_msg,
    response_code
FROM (
    SELECT
        domain,
        inventory_type,
        repr_error_msg,
        response_code,
        ROW_NUMBER() OVER (
            PARTITION BY domain, inventory_type
            ORDER BY repr_error_msg
        ) AS rn
    FROM err_tbl
)
WHERE rn = 1
)



SELECT 
pubs_data.publisher
,pubs_data.org_id
,pubs_data.inventory_type
,pubs_data.relationship_type_given
,pubs_data.account_manager_name
,pubs_data.account_manager_email
,pubs_data.domain
,pubs_data.ad_request
,pubs_data.revenue
,pubs_data.ads_txt_lines
,pubs_data.ssp_domain
,pubs_data.pub_id
,pubs_data.relationship_type
,pubs_data.ssp_id
,scraped.ads_txt_lines AS ads_txt_lines_scraped
,scraped.domain AS domain_scraped
,scraped.inventory_type AS inventory_type_scraped
,scraped.ads_txt_url AS ads_txt_url_scraped
,scraped.ssp_domain AS ssp_domain_scraped
,scraped.pub_id AS pub_id_scraped
,scraped.relationship_type AS relationship_type_scraped
,scraped.ssp_id AS ssp_id_scraped
,CASE WHEN scraped.ads_txt_lines IS NULL THEN 'missing' ELSE 'present' END AS ads_txt_status
,CASE WHEN error_tbl.repr_error_msg IS NOT NULL THEN 'failure' WHEN scraped.ads_txt_lines IS NULL THEN 'missing' ELSE 'present' END AS ads_txt_status_v2
,error_tbl.repr_error_msg
,error_tbl.response_code
FROM cleaned_pubs_data AS pubs_data LEFT JOIN cleaned_bronze_data AS scraped
ON pubs_data.pub_id = scraped.pub_id
AND pubs_data.inventory_type = scraped.inventory_type
AND pubs_data.domain = scraped.domain
AND pubs_data.ssp_domain = scraped.ssp_domain

LEFT JOIN error_tbl AS error_tbl
ON pubs_data.inventory_type = error_tbl.inventory_type
AND pubs_data.domain = error_tbl.domain

    """
)

In [ ]:
output_tbl.show()

In [6]:
output_tbl.to_csv("adhoc_BD_run.csv",overwrite=True, header=True)
